## Access Memory 访问内存

### 工具

#### 使用工具读取短期记忆 `ToolRuntime`

`runtime`使用参数（输入为）在工具中访问短期记忆（状态）`ToolRuntime`。

##### 状态注入（State Injection）与 `ToolRuntime`

在工具调用（Tool Call）中如何让工具**静默地**获取外部上下文。

以下这段代码演示了如何让一个 `tool`（工具）直接读取 Agent 的 `State`（状态），而**不需要模型在对话中显式地提供参数**。

- **常规做法**：模型必须在对话里看到 `user_id`，然后把它作为参数传给工具，如 `get_user_info(user_id="user_123")`。
- **本代码做法**：模型只发出了指令“查询用户信息”，它甚至不知道 `user_id` 的存在。工具通过 `ToolRuntime` 直接从后台的 `CustomState` 中“偷”到了这个 ID。

In [ ]:
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime


class CustomState(AgentState):
    user_id: str

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_user_info],
    state_schema=CustomState,
)

result = agent.invoke({
    "messages": "look up user information",
    "user_id": "user_123"
})

print(result["messages"][-1].content)

The user information has been found. The user's name is John Smith. Is there anything else you need to know?


###### 执行结果分析

1. 模型确实“看不见” `user_id`

在 `agent.invoke` 输入里，`user_id` 是放在字典里的。但是，模型输出的内容是：

> "The user information has been found. The user's name is John Smith..."

模型并没有说：“好的，我根据你提供的 `user_123` 查到了...” 这说明模型从头到尾都不知道 `user_id` 是什么。它只知道它执行了 `get_user_info` 这个动作，然后工具返回了“John Smith”。

2. 工具成功从后台“偷”到了数据

如果工具没有通过 `ToolRuntime` 拿到 `user_id`，它会返回 "Unknown user"。既然结果是 "John Smith"，说明：

- `CustomState` 里的 `user_id` 正确地通过 `agent.invoke` 传入了。
- `get_user_info` 工具成功在后台读取了状态，而模型对此全程处于“不知情”状态。

虽然 `State` 对模型不可见，但 **`State` 里的 `messages` 列表是模型唯一能看到的 `State` 字段**。

当 `get_user_info` 工具运行完后，它返回的结果（"User is John Smith"）会被包装成一条 `ToolMessage` 塞进 `messages` 列表。**只有在这一刻，关于 John Smith 的信息才正式进入了模型的“视野（Context）”里。**

**所以说：数据在 `State` 里时，模型是瞎子；数据变成 `Message` 后，模型才睁开了眼。** 

#### 从工具写入短期记忆

工具可以根据“**模型不可见但工具可见**”的 `Context`，去修改“**模型看不见**”的 `State`。

要在执行期间修改代理的短期记忆（状态），您可以直接从工具返回状态更新。这对于保存中间结果或使信息可供后续工具或提示访问非常有用。

| **数据层级** | **角色**       | **模型能看见吗？** | **谁能看见？**                         |
| ------------ | -------------- | ------------------ | -------------------------------------- |
| **Messages** | 视野           | **完全可见**       | 模型、工具、用户                       |
| **Context**  | 外部传入的机密 | **完全不可见**     | **仅限工具**（通过 `runtime.context`） |
| **State**    | 内部持久化数据 | **完全不可见**     | **仅限工具**（通过 `runtime.state`）   |


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command
from pydantic import BaseModel


class CustomState(AgentState):
    user_name: str


class CustomContext(BaseModel):
    user_id: str


@tool
def update_user_info(
    runtime: ToolRuntime[CustomContext, CustomState],
) -> Command:
    """Look up and update user info."""
    user_id = runtime.context.user_id
    name = "John Smith" if user_id == "user_123" else "Unknown user"
    return Command(
        update={
            "user_name": name,
            # update the message history
            "messages": [
                ToolMessage(
                    "Successfully looked up user information",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


@tool
def greet(runtime: ToolRuntime[CustomContext, CustomState]) -> str | Command:
    """Use this to greet the user once you found their info."""
    user_name = runtime.state.get("user_name", None)
    if user_name is None:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        "Please call the 'update_user_info' tool it will get and update the user's name.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    return f"Hello {user_name}!"


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[update_user_info, greet],
    state_schema=CustomState,
    context_schema=CustomContext,  # 这里的 Schema 是给代码校验用的，不是给模型读的
)

agent.invoke(
    {"messages": [{"role": "user", "content": "greet the user"}]},
    context=CustomContext(user_id="user_123"),
)

print(result["messages"][-1].content)

The user information has been found. The user's name is John Smith. Is there anything else you need to know?


##### 代码示例

代码的运行逻辑：

1. **模型是瞎子**：你调用 `agent.invoke` 时，模型只接到了一个指令：“向用户打招呼”。它不知道 `user_id` 是什么（因为它在 `Context` 里），也不知道 `user_name` 是什么（因为它在 `State` 里）。
2. **第一次碰撞（greet）**：模型尝试调用 `greet` 工具。
   - `greet` 工具检查 `State`，发现 `user_name` 为空。
   - 工具并没有报错，而是发出了一个 `Command` 告诉模型：“去调用 `update_user_info`”。
3. **核心魔法（update_user_info）**：
   - 该工具读取了 **`Context`** 里的 `user_id="user_123"`。
   - 它在后台查询到了名字叫 "John Smith"。
   - **最关键的一步**：它返回 `Command(update={"user_name": name})`。这行代码**修改了 State**，但模型依然看不见这个变量。它只看见了一条 `ToolMessage` 说“查询成功”。
4. **最终结果**：模型再次调用 `greet`，此时工具从 `State` 拿到了名字，返回 "Hello John Smith!"。

###### 代码分析

**完全的隐私屏蔽**：模型自始至终没见过用户的 ID，也没见过用户的真实姓名存在变量里，它只是像个接线员一样在调度工具。

**Context 是“阅后即焚”的**：`Context` 通常随请求传入（如 API 调用时的 Token），它不保存在 `checkpointer`（数据库）里。

**State 是“持久化”的**：一旦 `update_user_info` 更新了 `State`，以后这个 `thread_id` 下的对话，模型都能通过工具直接拿到名字，不需要再次查询。

##### 模型到底是怎么工作的？（以代码为例）

我们可以把这个过程想象成一个**盲人（模型）**、一个**导盲犬（工具）和一个盲文板（Messages）**：

1. **开始**：盲文板上写着“请打招呼”。
2. **动作**：盲人（模型）摸到了这条指令，但他不知道对方是谁。于是他决定拍拍导盲犬（调用 `greet` 工具）。
3. **工具逻辑**：导盲犬（工具）逻辑里写着：*“如果后台状态里没名字，就告诉盲人去查”*。它给盲人回了个信号。
4. **关键点（update_user_info）**：盲人（模型）收到了信号，决定调用 `update_user_info`。
   - **重点来了：** 盲人（模型）发起这个调用时，他并不知道自己正在利用 `user_id`。他只是按下了那个叫“更新信息”的按钮。
   - **幕后操作：** 工具在运行的时候，偷偷从后台的 `Context` 抽屉里拿出了 `user_id="user_123"`，查到了名字，然后把它塞进了后台的 `State` 抽屉。
5. **结果**：盲人（模型）收到一条反馈消息：“信息查好了”。他再次按下“打招呼”按钮，导盲犬（工具）从 `State` 抽屉拿到了名字，告诉盲人：“你可以说 Hello John Smith 了”。

### Prompt （以中间件MiddleWare引入）

在中间件中访问短期记忆（状态），以根据**对话历史记录**或**自定义状态字段**创建动态提示。

LangChain 中的一个核心高级特性：**动态系统提示词（Dynamic System Prompting）**

说明了如何利用 **模型不可见** 的后台上下文（Context），在**模型开始思考前**，动态地为它“定制”一副人设眼镜。

#### 1. 核心功能：Middleware（中间件）的介入

这段代码的核心在于 `middleware=[dynamic_system_prompt]`。

在传统的 Agent 开发中，系统提示词（System Prompt）通常是静态的硬编码。而这里的 `@dynamic_prompt` 装饰器起到了“**拦截器**”的作用：

1. **拦截**：当用户发送消息后，在消息到达模型（LLM）之前，中间件先跳出来。
2. **提取**：它从 `request.runtime.context` 中提取出 `user_name="John Smith"`。
3. **注入**：它把名字拼接到提示词里，生成 `Address the user as John Smith`。
4. **放行**：最后才把这个新鲜出炉的提示词和用户的提问一起发给模型。

#### 2. 这解决了什么问题？

“为什么不直接把名字写在初始消息里？” 这种做法有三个显著优势：

- **真正的“零感知”个性化**：模型在 `messages` 历史里看不到 `John Smith` 这个输入，但因为它被注入到了系统人设中，模型会像“本能”一样称呼你。
- **状态与展示分离**：`Context` 可以是加密的、动态变化的（比如根据时间段变成“早安/晚安”），而业务代码（`agent.invoke`）不需要每次都去手动修改提示词。
- **隐私保护**：敏感信息（如权限等级、用户真名）被锁在 `Context` 这个安全区，只在模型推理的一瞬间被注入，不会污染持久化的对话历史（Messages）。

### 3. 运行流程拆解

1. **用户提问**：“What is the weather in SF?”

2. **中间件运行**：发现 `context` 里有 `John Smith`，生成 System Message。

3. **模型接收到的真实 Prompt**：

   > **System:** You are a helpful assistant. Address the user as **John Smith**. **User:** What is the weather in SF?

4. **模型输出**：因为它看到了系统指令，它会回答类似：“Hello **John Smith**, the weather in SF is...”



这一招非常适合用来做：

- **多语言切换**：根据 `Context` 里的语言设置，动态要求模型用中文或英文回答。
- **风格切换**：根据 `Context` 里的用户偏好，让模型切换成“专业导师”或“幽默损友”模式。

In [6]:
from langchain.agents import create_agent
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class CustomContext(TypedDict):
    user_name: str


def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is always sunny!"


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context["user_name"]
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[dynamic_system_prompt],
    context_schema=CustomContext,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    context=CustomContext(user_name="John Smith"),
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in SF?
================================== Ai Message ==================================
Tool Calls:
  get_weather (b812b4c4-3a44-4742-8491-1f02287ab2ad)
 Call ID: b812b4c4-3a44-4742-8491-1f02287ab2ad
  Args:
    city: SF
================================= Tool Message =================================
Name: get_weather

The weather in SF is always sunny!
================================== Ai Message ==================================

The weather in SF is always sunny!



###### 代码分析

可以把它拆解为以下三个核心动作：

1. “截获与映射” (The Interception)

在 `agent.invoke` 被触发后，数据并没有立刻飞向模型，而是先进入了 **Middleware（中间件）** 的流水线。

- **数据来源**：`invoke` 时传入的 `context`。
- **处理逻辑**：中间件里的函数逻辑（即 `@dynamic_prompt` 装饰的函数）。
- **映射产物**：将原本冷冰冰的后台数据（如 `user_name: "John Smith"`）转换成了极具引导性的自然语言指令。

2. “隐形组装” (The Invisible Assembly)

在发给模型（Ollama/Qwen）的最终请求包里，消息列表被动态地扩充了。

- **最终结构**：`[ 动态生成的 SystemMessage ] + [ 原始的 messages 列表 ]`。
- **对模型而言**：它拿到的就是一个完整的、带有预设人设的任务说明。它并不知道这个 System Message 是刚刚才被“临时拼凑”出来的。

3. “预设角色” (Pre-setting the Persona)

这一步的作用就是你说的“预设角色”。它的妙处在于**“解耦”**：

- **用户层**（调用 `invoke` 的人）：只需要传最基础的数据（谁在聊天？当前什么环境？）。
- **逻辑层**（中间件）：定义如何把这些数据变成规则。
- **模型层**（LLM）：只需要在被赋予的角色下，处理当下的 `UserMessage`。

###### 💡 为什么这种“预设”比直接写死更高级？

可以把这种模式对比成**“给演员发剧本”**：

- **静态 Prompt**：相当于剧本是印死的，演员（模型）演谁都一样。
- **动态 Prompt (这段代码)**：相当于演员在后台化妆时，导演根据今天来现场的观众（`Context`），临时在剧本第一页写上：“今天的观众是 John Smith，请务必表现得亲切一些”。

**结果：** 演员（模型）上台后，看到的剧本已经更新了，他自然而然地就能以正确的角色身份去处理舞台上（`messages`）发生的事。

### before model

在中间件中访问短期记忆（状态）[`@before_model`](https://reference.langchain.com/python/langchain/agents/middleware/types/before_model)以在模型调用之前处理消息。

及时修剪上下文

 **“即时上下文修剪（JIT Context Trimming）”**。它与之前“摘要”或“物理删除”最大的区别在于：它发生在 **模型请求的前一刻**，而且它巧妙地利用了 `RemoveMessage` 的一种“大招”用法。

**这些消息确实存储在 `State` 中**。

但在 LangGraph 的体系里，我们要区分两个概念：

- **持久化 State**：存储在 `checkpointer`（即 `InMemorySaver` 或 `SqliteSaver`）里的完整历史。
- **运行时 State**：中间件 `trim_messages` 接收到的 `state` 参数，它是从数据库里读出来、准备喂给模型的“草稿”。

In [7]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model,
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob!


##### 代码分析

这段代码最狠的地方在这里：

```Python
"messages": [
    RemoveMessage(id=REMOVE_ALL_MESSAGES),
    *new_messages
]
```

这里的逻辑不是一条条去删，而是：

1. **清空（Nuke）**：使用 `REMOVE_ALL_MESSAGES` 瞬间清空当前会话在发送给模型前的所有消息缓存。
2. **重建（Rebuild）**：立刻把 `new_messages`（第一条 + 最近几条）塞回去。

**为什么要这么做？** 

因为直接计算哪些该留、哪些该删的 ID 非常麻烦。直接“全删再加回”是最稳健的修剪方式，能确保发给模型的消息列表绝对符合你的预期（只有第一条和最后几条）。

##### 拦截流程

这个中间件（`@before_model`）就像是一个**安检员**：

1. **用户调用** `agent.invoke`。
2. **准备阶段**：LangGraph 从 `InMemorySaver` 加载该 `thread_id` 的所有历史消息。
3. **拦截阶段（重点）**：在这些消息发往 Ollama 之前，`trim_messages` 拦截了它们。
   - 它发现消息太多了。
   - 它生成了一个指令：“先把内存里的全删了，只保留第一条和最近 3-4 条”。
4. **发送阶段**：修剪后的、精简的消息列表被发给 Ollama。
5. **存储阶段**：注意！这个修剪后的状态会被写回 `checkpointer`。

刚才跑通的代码其实展示了一个**工业级 Agent 的内存管理模版**：

1. **State (持久化存储)**：`InMemorySaver` 负责在大脑后台记下所有东西。
2. **Middleware (@before_model)**：像一个“过滤器”。在数据发给 Ollama 之前，它在**内存**里做了一次全量清理（`REMOVE_ALL_MESSAGES`），然后只挑出最重要的东西（开头 + 结尾）喂给模型。
3. **结果**：模型（Ollama）觉得很舒服，因为上下文非常干净，没有不认识的特殊消息类型，也没有冗长的废话。

这种“保留第一条 + 最近 N 条”的策略叫 **First-and-Last Trimming**。它非常适合：

- **第一条是 System Prompt 或核心设定**（比如你的名字、AI 的身份）。
- **最后几条是当下的对话上下文**。

### After Model 仿制模型

[`@after_model`](https://reference.langchain.com/python/langchain/agents/middleware/types/after_model)在模型调用后，中间件会访问短期记忆（状态）来处理消息。

#### 1. 核心功能：`@after_model` 的守卫角色

这段代码的逻辑流程如下：

1. **模型输出**：Ollama 生成了回答（AIMessage）。
2. **后置拦截**：在消息正式存入 `checkpointer`（数据库/内存）以及展示给用户之前，`validate_response` 函数被触发了。
3. **内容审查**：它检查 AI 的回复里是否包含了敏感词（如 `password` 或 `secret`）。
4. **物理抹除**：一旦发现“违禁词”，它立刻返回一个 `RemoveMessage`。

**结果：** 用户将永远看不到这条包含敏感信息的回复，且这条消息也不会在历史记录中留下痕迹。

#### 2. 为什么这段代码很重要？

这其实是构建 **安全合规（Compliance）** AI 的标准做法。在实际应用中，它可以演变成：

- **隐私脱敏**：如果 AI 不小心说出了用户的手机号或 ID，后置操作可以自动将其打码或删除。
- **事实校验（Hallucination Check）**：你可以调用另一个小模型来检查主模型的回答是否有误，如果有误，直接拦截。
- **格式修正**：如果 AI 返回的 JSON 格式不对，你可以在这里尝试修复，或者干脆删掉让它重写。

| **中间件类型**        | **触发时机** | **典型用途**                   |
| --------------------- | ------------ | ------------------------------ |
| **`@dynamic_prompt`** | 模型运行前   | 注入人设、动态背景。           |
| **`@before_model`**   | 模型运行前   | 上下文修剪、Token 控制。       |
| **`@after_model`**    | 模型运行后   | 安全审查、内容过滤、格式检查。 |

#### 3. 一个非常关键的细节：`last_message.id`

在这段代码里：

```Python
return {"messages": [RemoveMessage(id=last_message.id)]}
```

能成功运行的前提是 **模型返回的消息已经分配了 ID**。在 LangChain 的 `create_agent` 流程中，`after_model` 触发时，系统已经为这条刚产生的 `AIMessage` 预分配了 ID，所以我们可以精准地定位并“干掉”它。

In [8]:
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime


@after_model
def validate_response(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove messages containing sensitive words."""
    STOP_WORDS = ["password", "secret"]
    last_message = state["messages"][-1]
    if any(word in last_message.content for word in STOP_WORDS):
        return {"messages": [RemoveMessage(id=last_message.id)]}
    return None

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[validate_response],
    checkpointer=InMemorySaver(),
)